# ClearML Dataset 多步顺序演示

本 Notebook 目标：用最少的业务代码，覆盖尽量多的 `from clearml import Dataset` 核心能力（创建/上传、获取/下载、任务引用 alias、父子版本、增量更新、目录同步、发布）。

运行前提：
- 你已经配置好 ClearML Server 与凭证（例如已运行过 `clearml-init`）。
- 你的 ClearML 配置中有可用的默认存储（Dataset `upload()` 需要），或你在服务端/配置里设置了默认 output/storage。

注意：本示例会在 ClearML Web UI 中创建一些 Demo 对象（Task、Dataset 版本）。

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path
from uuid import uuid4

def tree(path: str, max_lines: int = 200):
    """打印目录结构（辅助展示），与 ClearML 核心功能无关"""
    p = Path(path)
    lines = []
    for root, dirs, files in os.walk(p):
        root_p = Path(root)
        rel = root_p.relative_to(p)
        indent = "  " * (0 if rel == Path('.') else len(rel.parts))
        lines.append(f"{indent}{rel}/")
        for f in sorted(files):
            lines.append(f"{indent}  - {f}")
        if len(lines) >= max_lines:
            lines.append("...")
            break
    print("\n".join(lines))

run_id = uuid4().hex[:8]
project_name = "demo_project"
dataset_name = f"demo_dataset_{run_id}"
task_project = "train_project"
task_name = f"dataset_demo_{run_id}"

workdir = Path(tempfile.mkdtemp(prefix=f"clearml_dataset_demo_{run_id}_"))
print("workdir =", str(workdir))

## 0) 环境检查 + 初始化 Task

ClearML 的 Dataset 也属于“需要服务端记录”的对象。为了让后续演示里的 `alias=...` 生效，我们先初始化一个 Task。

如果这里报错：
- 一般是没配置 ClearML 凭证/Server（先运行 `clearml-init`）
- 或网络/Server 不可达

In [ ]:
try:
    from clearml import Task, Dataset
except Exception as e:
    raise RuntimeError(
        "没有安装 clearml 包。请先在当前环境安装：pip install clearml\n"
        f"原始错误: {e}"
    )

# 重点：为了让模型/文件等能够自动上传，通常建议 output_uri=True
# 这里我们只演示 Dataset，所以 Task 只用于绑定 alias（记录“任务依赖了哪个数据集”）
task = Task.init(
    project_name=task_project,
    task_name=task_name,
    output_uri=True,
)
print("Task id =", task.id)

## 1) 生成一份最小的本地数据

我们用临时目录创建几个小文件，模拟“本地数据目录”。（这些文件仅用于演示 `add_files()`/`sync_folder()`）

In [ ]:
local_data_v1 = workdir / "local_data_v1"
local_data_v1.mkdir(parents=True, exist_ok=True)

(local_data_v1 / "README.txt").write_text("demo dataset v1\n", encoding="utf-8")
(local_data_v1 / "a.txt").write_text("a=1\n", encoding="utf-8")
(local_data_v1 / "b.txt").write_text("b=2\n", encoding="utf-8")

sub = local_data_v1 / "sub"
sub.mkdir(exist_ok=True)
(sub / "c.txt").write_text("c=3\n", encoding="utf-8")

print("local_data_v1:")
tree(str(local_data_v1))

## 2) 创建并上传一个 Dataset 版本（create / add_files / upload / finalize）

这一段覆盖 Dataset 最核心生命周期：

```python
Dataset.create() -> add_files() -> upload() -> finalize()
```

关键点：
- `add_files()` 只是“登记”本地文件
- `upload()` 才会把本地文件上传到远端存储
- `finalize()` 会冻结当前版本，后续才好稳定复现与派生

In [ ]:
ds_v1 = Dataset.create(
    dataset_project=project_name,
    dataset_name=dataset_name,
)

# 重点：把整个目录登记进 Dataset
# dataset_path 可以理解为“在数据集内部的相对路径前缀”，便于结构化组织
ds_v1.add_files(str(local_data_v1), dataset_path="data")

# 重点：upload 依赖 ClearML 的默认存储配置
ds_v1.upload()

# 重点：finalize 冻结版本（稳定复用/父子继承的前提）
ds_v1.finalize()

print("ds_v1.id =", ds_v1.id)
print("ds_v1.project =", ds_v1.project)
print("ds_v1.name =", ds_v1.name)
print("ds_v1.version =", ds_v1.version)
print("ds_v1.tags =", getattr(ds_v1, "tags", None))

## 3) 在“训练/消费代码”里使用已有 Dataset（get / get_local_copy）

你通常只需要两步：
- `Dataset.get(...)` 定位数据集
- `get_local_copy()` 下载到 ClearML 缓存目录，返回一个只读路径

这种方式适合训练场景：代码只读消费数据，不在缓存目录里修改文件。

In [ ]:
ds_readonly = Dataset.get(
    dataset_project=project_name,
    dataset_name=dataset_name,
)

local_cache_path = ds_readonly.get_local_copy()
print("local_cache_path =", local_cache_path)
print("cache content:")
tree(local_cache_path)

## 4) 让 Task 明确记录“用了哪份数据”（alias）

ClearML 推荐你在 `Task.init()` 之后，使用：

```python
Dataset.get(..., alias="train_data", overridable=True)
```

这样当前 Task 会记录对 Dataset 的引用，后续远端执行/复现更清晰。

In [ ]:
ds_with_alias = Dataset.get(
    dataset_project=project_name,
    dataset_name=dataset_name,
    alias="train_data",
    overridable=True,
)

alias_path = ds_with_alias.get_local_copy()
print("alias_path =", alias_path)
print("当前 Task 已记录数据集引用（去 Web UI 的 Task 页面通常能看到）")

## 5) 拿到一份“可写”的本地副本（get_mutable_local_copy）

`get_local_copy()` 返回的是缓存目录里的只读副本；如果你想在本地继续加工（比如解压/生成新文件），建议用：

```python
get_mutable_local_copy(target_folder=..., overwrite=True)
```

这一步常用在“数据加工 -> 再注册成新 Dataset 版本”的场景。

In [ ]:
mutable_folder = workdir / "mutable_copy"
mutable_path = ds_readonly.get_mutable_local_copy(
    target_folder=str(mutable_folder),
    overwrite=True,
)
print("mutable_path =", mutable_path)

# 重点：这里的文件可以写（演示修改一行）
p = Path(mutable_path) / "data" / "a.txt"
p.write_text("a=100\n", encoding="utf-8")
print("修改后 a.txt 内容:")
print(p.read_text(encoding="utf-8"))

## 6) 基于旧版本派生新版本（parent/child）

最典型的“增量更新”流程：
- 先 `get` 旧版本
- 再 `create(parent_datasets=[parent.id])` 生成子版本
- 然后新增/删除/同步
- 最后 `upload()` + `finalize()`

下面演示：在子版本里新增一个文件、再删除一个文件。

In [ ]:
parent = Dataset.get(dataset_project=project_name, dataset_name=dataset_name)

ds_v2 = Dataset.create(
    dataset_project=project_name,
    dataset_name=dataset_name,
    parent_datasets=[parent.id],
)

# 新增：创建一个新文件并加到 v2
local_inc = workdir / "increment"
local_inc.mkdir(exist_ok=True)
(local_inc / "new.txt").write_text("new file in v2\n", encoding="utf-8")
ds_v2.add_files(str(local_inc), dataset_path="data")

# 删除：从 v2 中删除某个路径（这里删除 v1 的 b.txt）
# 重点：remove_files 的参数是“数据集内部路径”（和你 add_files 时的 dataset_path 对应）
ds_v2.remove_files("data/b.txt")

ds_v2.upload()
ds_v2.finalize()

print("ds_v2.id =", ds_v2.id)
print("ds_v2.version =", ds_v2.version)

## 7) 把“本地目录的真实状态”同步到 Dataset（writable_copy + sync_folder）

当你的本地目录就是最终结果（比如数据加工产物都在一个目录里），`sync_folder()` 很方便：
- 它按目录的真实状态做“增量同步”
- 适合做“目录级别的一致性更新”

这里演示：
- 基于现有数据集创建 `writable_copy=True` 的可写子版本
- 准备一个新目录作为“当前真实状态”
- 用 `sync_folder(local_path=..., dataset_path=...)` 同步进去
- 再 `upload()` + `finalize()`

In [ ]:
ds_writable = Dataset.get(
    dataset_project=project_name,
    dataset_name=dataset_name,
    writable_copy=True,
)

current_state = workdir / "current_state"
current_state.mkdir(exist_ok=True)
(current_state / "x.txt").write_text("x=10\n", encoding="utf-8")
(current_state / "y.txt").write_text("y=20\n", encoding="utf-8")

# 重点：把 current_state 目录同步到数据集内部的 synced/ 路径
ds_writable.sync_folder(
    local_path=str(current_state),
    dataset_path="synced",
)

ds_writable.upload()
ds_writable.finalize()

print("ds_synced.id =", ds_writable.id)
print("ds_synced.version =", ds_writable.version)

## 8) （可选）发布一个稳定版本（publish）

`publish()` 用于把某个版本标记为“稳定可用”。不同团队对它的使用习惯不同：
- 有的团队只 publish 经过验证的数据版本
- 有的团队用 tag/版本号做管理

如果你的 ClearML 版本不支持 `publish()`，这一格会自动跳过。

In [ ]:
if hasattr(ds_v2, "publish"):
    ds_v2.publish()
    print("已调用 ds_v2.publish()")
else:
    print("当前 clearml 版本的 Dataset 没有 publish()，已跳过")

## 9) 最后：下载并验证某个版本内容

为了确认版本确实发生了变化，我们分别下载一个 v2 或 sync 版本，看看目录里是否包含我们预期的文件。

In [ ]:
ds_latest = Dataset.get(dataset_id=ds_writable.id)
latest_path = ds_latest.get_local_copy()
print("latest_path =", latest_path)
tree(latest_path)

## 10) 清理临时目录（可选）

仅清理本机临时工作目录，不会删除 ClearML Server 上已注册的数据集版本。

In [ ]:
# 如果你希望保留本地临时文件用于排查，可以注释掉这段
shutil.rmtree(workdir, ignore_errors=True)
print("已清理 workdir")